# Quickstart: Hetarynes_Pipeline in 5 minutes

This notebook takes 10 sample heteroaromatic SMILES through **Modules 1 → 3** of
the [Hetarynes_Pipeline](../README.md) and renders the resulting arynes — all
on a laptop, with **no DFT calculations**, in under 30 seconds.

The pipeline does the following:

1. **Module 1** — validate & canonicalize SMILES, deduplicate.
2. **Module 2** — SMARTS substructure search for [hetero]aromatic cores containing
   at least one internal `C=C` bond (the aryne precursor site).
3. **Module 3** — Reaction SMARTS converts `C=C → C#C` to generate aryne products.

To run the *full* pipeline including DFT geometry optimizations and descriptor
extraction, see [Module 4's README](../Module4_Generate_Conformers_Write_DFT_Inputs/README.md)
onward. The full pipeline requires Orca 5.0.3 and HPC access.

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from rdkit.Chem.Draw import IPythonConsole

## Step 1 — Sample SMILES

Ten common heteroaromatic ring systems — mix of 5- and 6-membered, with N / O / S
heteroatoms. In a real run, these come from CSV files in
[`csv_backups/input_csv_datasets/`](../csv_backups/input_csv_datasets/).

In [ ]:
sample_smiles = [
    "c1ccncc1",          # pyridine
    "c1ccnnc1",          # pyridazine
    "c1cncnc1",          # pyrimidine
    "c1ccoc1",           # furan
    "c1ccsc1",           # thiophene
    "c1cc[nH]c1",        # pyrrole
    "c1cnc[nH]1",        # imidazole
    "c1ocnc1",           # oxazole
    "c1scnc1",           # thiazole
    "c1ccc2[nH]ccc2c1",  # indole
]
print(f"Starting with {len(sample_smiles)} SMILES")

## Step 2 — Canonicalize and deduplicate (Module 1 equivalent)

Each SMILES is parsed into an RDKit Mol and re-emitted in canonical form so that
string-based deduplication actually catches duplicates (the same molecule written
two different ways).

`dict.fromkeys()` preserves insertion order — this is the same trick Module 2
uses for deterministic ID assignment across runs.

In [ ]:
canonical = []
for smi in sample_smiles:
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        canonical.append(Chem.MolToSmiles(mol))
    else:
        print(f"  failed to parse: {smi}")

unique = list(dict.fromkeys(canonical))
print(f"\nAfter validation + dedup: {len(unique)} SMILES")
for s in unique:
    print(f"  {s}")

## Step 3 — Filter for cores with a convertible C=C (Module 2 equivalent)

The aryne precursor site is an aromatic C–C with no substituents on either
carbon (so a hydrogen can leave from each). In SMARTS, that's two adjacent
aromatic carbons each with degree 2 (one hydrogen, one ring-neighbor):

```
[cH1D2;R][cH1D2;R]
```

`[cH1D2;R]` = aromatic carbon (`c`), exactly 1 H (`H1`), degree 2 (`D2`), in a
ring (`R`). Module 2 uses this exact SMARTS to identify viable arene cores.

In [ ]:
ALKENE_SMARTS = Chem.MolFromSmarts('[cH1D2;R][cH1D2;R]')

eligible = [
    s for s in unique
    if Chem.MolFromSmiles(s).GetSubstructMatches(ALKENE_SMARTS)
]
ineligible = [s for s in unique if s not in eligible]

print(f"Cores with at least one convertible C=C: {len(eligible)} / {len(unique)}")
print("\nEligible:")
for s in eligible: print(f"  {s}")
if ineligible:
    print("\nDropped (no convertible C=C):")
    for s in ineligible: print(f"  {s}")

### Visualize the eligible arene cores

In [ ]:
Draw.MolsToGridImage(
    [Chem.MolFromSmiles(s) for s in eligible],
    molsPerRow=5,
    subImgSize=(200, 200),
    legends=[f"arene_{i+1}" for i in range(len(eligible))],
)

## Step 4 — Generate arynes via Reaction SMARTS (Module 3 equivalent)

The aryne reaction is *"replace an aromatic C–C single bond with an aromatic
C≡C triple bond"*. In Reaction SMARTS, mapped atoms `[:1]` and `[:2]` on the
LHS appear in the same positions on the RHS, now linked by `#` (triple bond):

```
[cH1D2;R:1][cH1D2;R:2] >> [c:1]#[c:2]
```

Module 3 runs this reaction on every eligible arene in the input. Here we
take the **first eligible arene** from Step 3 and enumerate every possible
aryne it can produce.

In [ ]:
ARYNE_REACTION = AllChem.ReactionFromSmarts(
    '[cH1D2;R:1][cH1D2;R:2]>>[c:1]#[c:2]'
)

arene_example = eligible[0]
arene_mol = Chem.MolFromSmiles(arene_example)
print(f"Generating arynes from: {arene_example}\n")

aryne_smiles = []
for products in ARYNE_REACTION.RunReactants((arene_mol,)):
    for product_mol in products:
        # Arynes contain an aromatic-triple-bond which isn't a valence-allowed
        # structure, so RDKit's full sanitizer rejects them. We emit the SMILES
        # without sanitizing — same approach Module 3 uses.
        smi = Chem.MolToSmiles(product_mol)
        aryne_smiles.append(smi)

# Deduplicate symmetric duplicates from the SMARTS match
aryne_smiles = list(dict.fromkeys(aryne_smiles))
print(f"Generated {len(aryne_smiles)} unique aryne products:\n")
for s in aryne_smiles:
    print(f"  {s}")

### Visualize the generated arynes

Note: arynes contain a formal aromatic triple bond (lowercase `c#c`), which is
not a normal SMILES construct — we pass `sanitize=False` to RDKit so it draws
the structure without complaining about valence.

In [ ]:
aryne_mols = [Chem.MolFromSmiles(s, sanitize=False) for s in aryne_smiles]
Draw.MolsToGridImage(
    aryne_mols,
    molsPerRow=5,
    subImgSize=(220, 220),
    legends=[f"aryne_{i+1}" for i in range(len(aryne_mols))],
)

## What's next?

You just ran the SMILES-processing front of the pipeline (Modules 1 → 3). The
full HAL-8000 workflow continues with:

- [**Module 4**](../Module4_Generate_Conformers_Write_DFT_Inputs/README.md) —
  SMILES → 3D coordinates → Orca DFT input files. Requires AQME (`csearch`,
  `qprep`) and OpenBabel.
- [**Module 5**](../Module5_Process_DFT_Output_Files/README.md) — DFT output
  validation and triage of failed calculations. Requires HPC access to run the
  DFT calculations between Modules 4 and 5.
- [**Module 6**](../Module6_Extract_Dehydrogenation_Energies/README.md) —
  extract optimized geometries and compute ΔE(arene → aryne + H₂).
- [**Module 7**](../Module7_Extract_DFT_Molecular_Descriptors/README.md) —
  extract ~15 quantitative + qualitative descriptors per aryne.
- [**Module 8**](../Module8_Data_Analysis_and_Visualization/) — analysis and
  ML modeling (histograms, univariate scatter, Random-Forest regression).

The HAL-8000 dataset itself contains 7,143 arenes and 8,116 arynes
computed at M06-2X-D3 / def2-SVP. See the [main README](../README.md) for the
dataset description and citation information.